# Sprint 3 — Otimização de Modelos e Avaliação Avançada

**Projeto:** Sistema de Predição de Evasão Escolar  
**Objetivo:** Otimizar hiperparâmetros dos modelos da Sprint 2 e realizar avaliação avançada com validação cruzada, curvas de aprendizado, calibração e curvas Precisão-Recall.

---
**Etapas:**
1. Carregamento e pré-processamento
2. GridSearchCV — Regressão Logística
3. GridSearchCV — Random Forest
4. Comparação Base vs Otimizado
5. Validação Cruzada Estratificada (5-Fold)
6. Curvas de Aprendizado
7. Curva Precisão-Recall
8. Calibração dos Modelos
9. Importância de Features — RF Otimizado
10. Análise de Erros
11. Conclusões e Recomendações
12. Salvar Modelos Otimizados

## 1. Carregamento e Pré-processamento

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_validate, learning_curve, train_test_split
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
    precision_recall_curve, average_precision_score,
)
from sklearn.calibration import calibration_curve

from preprocessamento import executar_preprocessamento

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')

CORES = {'Regressão Logística': '#8e44ad', 'Random Forest': '#e67e22'}

print('Bibliotecas carregadas com sucesso.')

In [ ]:
X_train, X_test, y_train, y_test, preprocessador, nomes_features = executar_preprocessamento()

# Modelos base da Sprint 2
BASE_DIR = os.path.abspath('..')
lr_base = joblib.load(os.path.join(BASE_DIR, 'models', 'modelo_regressao_logistica.pkl'))
rf_base = joblib.load(os.path.join(BASE_DIR, 'models', 'modelo_random_forest.pkl'))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Treino: {len(X_train)} amostras | Teste: {len(X_test)} amostras')
print(f'Features: {len(nomes_features)}')

## 2. GridSearchCV — Regressão Logística

In [ ]:
grade_lr = {
    'C':            [0.01, 0.1, 1, 10, 100],
    'solver':       ['lbfgs', 'liblinear'],
    'penalty':      ['l2'],
    'max_iter':     [1000],
    'random_state': [42],
}

grid_lr = GridSearchCV(
    LogisticRegression(), grade_lr,
    scoring='roc_auc', cv=cv, n_jobs=-1, verbose=1,
)
grid_lr.fit(X_train, y_train)

print(f'\nMelhores parâmetros : {grid_lr.best_params_}')
print(f'Melhor ROC-AUC (CV) : {grid_lr.best_score_:.4f}')

In [ ]:
# Visualizar resultados do GridSearch — LR
resultados_grid_lr = pd.DataFrame(grid_lr.cv_results_)
resultados_grid_lr = resultados_grid_lr[['param_C', 'param_solver', 'mean_test_score', 'std_test_score']]
resultados_grid_lr.columns = ['C', 'Solver', 'ROC-AUC Médio', 'Desvio']
resultados_grid_lr = resultados_grid_lr.sort_values('ROC-AUC Médio', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
for solver in resultados_grid_lr['Solver'].unique():
    sub = resultados_grid_lr[resultados_grid_lr['Solver'] == solver]
    ax.errorbar(
        sub['C'].astype(str), sub['ROC-AUC Médio'],
        yerr=sub['Desvio'], marker='o', label=solver, capsize=4,
    )
ax.set_xlabel('C (regularização)')
ax.set_ylabel('ROC-AUC Médio (CV)')
ax.set_title('GridSearch — Regressão Logística')
ax.legend(title='Solver')
ax.set_ylim(0.97, 1.0)
plt.tight_layout()
plt.show()

## 3. GridSearchCV — Random Forest

In [ ]:
grade_rf = {
    'n_estimators':      [50, 100, 200],
    'max_depth':         [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'random_state':      [42],
}

grid_rf = GridSearchCV(
    RandomForestClassifier(), grade_rf,
    scoring='roc_auc', cv=cv, n_jobs=-1, verbose=1,
)
grid_rf.fit(X_train, y_train)

print(f'\nMelhores parâmetros : {grid_rf.best_params_}')
print(f'Melhor ROC-AUC (CV) : {grid_rf.best_score_:.4f}')

In [ ]:
# Heatmap de resultados — RF (n_estimators x max_depth)
res_rf = pd.DataFrame(grid_rf.cv_results_)
res_rf = res_rf[res_rf['param_min_samples_split'] == grid_rf.best_params_['min_samples_split']]

pivot = res_rf.pivot_table(
    index='param_max_depth',
    columns='param_n_estimators',
    values='mean_test_score',
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    pivot, annot=True, fmt='.4f', cmap='YlOrRd',
    linewidths=0.5, ax=ax,
)
ax.set_title(f'GridSearch RF — ROC-AUC (min_samples_split={grid_rf.best_params_["min_samples_split"]})')
ax.set_xlabel('n_estimators')
ax.set_ylabel('max_depth')
plt.tight_layout()
plt.show()

## 4. Comparação: Base vs Otimizado

In [ ]:
lr_opt = grid_lr.best_estimator_
rf_opt = grid_rf.best_estimator_

def metricas(modelo, X, y):
    yp    = modelo.predict(X)
    yprob = modelo.predict_proba(X)[:, 1]
    return {
        'Acurácia': round(accuracy_score(y, yp), 4),
        'Precisão': round(precision_score(y, yp, zero_division=0), 4),
        'Recall':   round(recall_score(y, yp, zero_division=0), 4),
        'F1-Score': round(f1_score(y, yp, zero_division=0), 4),
        'ROC-AUC':  round(roc_auc_score(y, yprob), 4),
    }

resultados = {
    'LR Base':      metricas(lr_base, X_test, y_test),
    'LR Otimizado': metricas(lr_opt,  X_test, y_test),
    'RF Base':      metricas(rf_base, X_test, y_test),
    'RF Otimizado': metricas(rf_opt,  X_test, y_test),
}

df_comp = pd.DataFrame(resultados).T
print('Comparação Base vs Otimizado:')
display(df_comp)

In [ ]:
metricas_plot = ['Acurácia', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metricas_plot))
largura = 0.2
cores_comp = ['#bdc3c7', '#8e44ad', '#f0b27a', '#e67e22']

fig, ax = plt.subplots(figsize=(10, 5))
for i, (nome, m) in enumerate(resultados.items()):
    vals = [m[k] for k in metricas_plot]
    bars = ax.bar(x + i * largura, vals, largura, label=nome, color=cores_comp[i])
    ax.bar_label(bars, fmt='%.4f', padding=2, fontsize=8)

ax.set_xticks(x + largura * 1.5)
ax.set_xticklabels(metricas_plot)
ax.set_ylim(0.88, 1.03)
ax.set_ylabel('Valor')
ax.set_title('Base vs Otimizado — Comparação de Métricas')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 5. Validação Cruzada Estratificada (5-Fold)

In [ ]:
scoring_cv = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
nomes_cv   = ['Acurácia', 'Precisão', 'Recall', 'F1-Score', 'ROC-AUC']

linhas_cv = []
for nome, modelo in [('Regressão Logística', lr_opt), ('Random Forest', rf_opt)]:
    scores = cross_validate(modelo, X_train, y_train, cv=cv, scoring=scoring_cv, n_jobs=-1)
    for chave, rotulo in zip(scoring_cv, nomes_cv):
        vals = scores[f'test_{chave}']
        linhas_cv.append({
            'Modelo':   nome,
            'Métrica':  rotulo,
            'Média':    round(vals.mean(), 4),
            'Desvio':   round(vals.std(), 4),
            'Min':      round(vals.min(), 4),
            'Max':      round(vals.max(), 4),
        })

df_cv = pd.DataFrame(linhas_cv)
display(df_cv.pivot(index='Métrica', columns='Modelo', values=['Média', 'Desvio']))

In [ ]:
# Boxplot dos scores de cada fold
fig, axes = plt.subplots(1, len(scoring_cv), figsize=(16, 4))
for ax, (chave, rotulo) in zip(axes, zip(scoring_cv, nomes_cv)):
    dados_box = []
    labels_box = []
    for nome, modelo in [('LR', lr_opt), ('RF', rf_opt)]:
        s = cross_validate(modelo, X_train, y_train, cv=cv, scoring=chave, n_jobs=-1)
        dados_box.append(s['test_score'])
        labels_box.append(nome)
    bp = ax.boxplot(dados_box, labels=labels_box, patch_artist=True)
    bp['boxes'][0].set_facecolor('#c39bd3')
    bp['boxes'][1].set_facecolor('#f0b27a')
    ax.set_title(rotulo)
    ax.set_ylim(0.85, 1.02)
plt.suptitle('Distribuição dos Scores por Fold (5-CV)', y=1.02)
plt.tight_layout()
plt.show()

## 6. Curvas de Aprendizado

In [ ]:
X_full = np.vstack([X_train, X_test])
y_full = np.concatenate([np.array(y_train), np.array(y_test)])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (nome, modelo) in zip(axes, [('Regressão Logística', lr_opt), ('Random Forest', rf_opt)]):
    tamanhos, scores_tr, scores_val = learning_curve(
        modelo, X_full, y_full,
        train_sizes=np.linspace(0.1, 1.0, 8),
        cv=cv, scoring='roc_auc', n_jobs=-1,
    )
    tr_media = scores_tr.mean(axis=1)
    tr_std   = scores_tr.std(axis=1)
    val_media = scores_val.mean(axis=1)
    val_std   = scores_val.std(axis=1)

    ax.plot(tamanhos, tr_media,  'o-', color='#2980b9', label='Treino')
    ax.fill_between(tamanhos, tr_media - tr_std,  tr_media + tr_std,  alpha=0.15, color='#2980b9')
    ax.plot(tamanhos, val_media, 's-', color=CORES[nome], label='Validação')
    ax.fill_between(tamanhos, val_media - val_std, val_media + val_std, alpha=0.15, color=CORES[nome])
    ax.set_xlabel('Tamanho do treino')
    ax.set_ylabel('ROC-AUC')
    ax.set_title(f'Curva de Aprendizado — {nome}')
    ax.set_ylim(0.88, 1.01)
    ax.legend()

plt.tight_layout()
plt.show()

## 7. Curva Precisão-Recall

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for nome, modelo in [('Regressão Logística', lr_opt), ('Random Forest', rf_opt)]:
    y_proba = modelo.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    ax.plot(rec, prec, lw=2.5, label=f'{nome}  (AP = {ap:.4f})', color=CORES[nome])

baseline = y_test.mean()
ax.axhline(y=baseline, color='gray', linestyle='--', lw=1.5, label=f'Baseline ({baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precisão')
ax.set_title('Curva Precisão-Recall')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 8. Calibração dos Modelos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (nome, modelo) in zip(axes, [('Regressão Logística', lr_opt), ('Random Forest', rf_opt)]):
    y_proba = modelo.predict_proba(X_test)[:, 1]
    frac_pos, media_prev = calibration_curve(y_test, y_proba, n_bins=8)
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Calibração perfeita')
    ax.plot(media_prev, frac_pos, 'o-', lw=2.5, color=CORES[nome], label=nome, markersize=7)
    desvio = float(np.mean(np.abs(frac_pos - media_prev)))
    ax.set_xlabel('Probabilidade prevista')
    ax.set_ylabel('Fração de positivos reais')
    ax.set_title(f'Calibração — {nome}\nDesvio médio: {desvio:.4f}')
    ax.legend(loc='upper left')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 9. Importância de Features — RF Otimizado

In [ ]:
df_imp = pd.DataFrame({
    'Feature':     nomes_features,
    'Importância': rf_opt.feature_importances_,
}).sort_values('Importância', ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(df_imp['Feature'], df_imp['Importância'], color='#e67e22', alpha=0.85)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
ax.set_xlabel('Importância (Gini)')
ax.set_title('Importância das Features — Random Forest Otimizado')
ax.set_xlim(0, df_imp['Importância'].max() * 1.18)
plt.tight_layout()
plt.show()

print('\nTop 3 features:')
print(df_imp.sort_values('Importância', ascending=False).head(3).to_string(index=False))

## 10. Análise de Erros

In [ ]:
# Reconstruir X_test como DataFrame para análise de erros
df_full = pd.read_csv(os.path.join(BASE_DIR, 'data', 'alunos_limpo.csv'))
X_df = df_full.drop(columns=['id_aluno', 'nome', 'evadiu'])
y_full_series = df_full['evadiu']
_, X_test_df, _, _ = train_test_split(X_df, y_full_series, test_size=0.2, random_state=42, stratify=y_full_series)

y_pred_lr_opt = lr_opt.predict(X_test)
y_proba_lr    = lr_opt.predict_proba(X_test)[:, 1]

df_erros = X_test_df.copy().reset_index(drop=True)
df_erros['y_real']     = np.array(y_test)
df_erros['y_pred']     = y_pred_lr_opt
df_erros['prob_evasao'] = y_proba_lr
df_erros['erro']       = df_erros['y_real'] != df_erros['y_pred']

# Falsos positivos vs falsos negativos
fp = df_erros[(df_erros['y_real'] == 0) & (df_erros['y_pred'] == 1)]
fn = df_erros[(df_erros['y_real'] == 1) & (df_erros['y_pred'] == 0)]

print(f'Falsos Positivos (classificado como evadiu, mas não evadiu): {len(fp)}')
print(f'Falsos Negativos (não classificado como evadiu, mas evadiu): {len(fn)}')

print('\nPerfil médio dos Falsos Negativos (erros mais críticos):')
print(fn[['cr', 'faltas', 'reprovacoes', 'prob_evasao']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['cr', 'faltas', 'reprovacoes']):
    ax.hist(fp[col], bins=10, alpha=0.6, color='#3498db', label='Falso Positivo')
    ax.hist(fn[col], bins=10, alpha=0.6, color='#e74c3c', label='Falso Negativo')
    ax.set_xlabel(col.upper())
    ax.set_ylabel('Frequência')
    ax.set_title(f'Distribuição de {col.upper()} nos Erros')
    ax.legend()
plt.suptitle('Análise de Erros — Regressão Logística Otimizada', y=1.02)
plt.tight_layout()
plt.show()

## 11. Conclusões e Recomendações

| Aspecto | Resultado |
|---|---|
| Melhor método de otimização | GridSearchCV + StratifiedKFold 5-fold |
| Modelo vencedor | Verificar célula abaixo |
| Ganho de ROC-AUC (LR) | Verificar célula abaixo |
| Ganho de ROC-AUC (RF) | Verificar célula abaixo |
| Estabilidade (CV std) | < 0.005 indica modelo estável |

**Recomendações:**
- Usar o modelo otimizado com melhor ROC-AUC e menor desvio no CV
- Monitorar Falsos Negativos (alunos em risco não identificados) — são os erros de maior custo institucional
- Considerar ajuste de threshold para aumentar Recall se o objetivo for máxima cobertura dos alunos em risco

In [ ]:
auc_lr_base = roc_auc_score(y_test, lr_base.predict_proba(X_test)[:, 1])
auc_rf_base = roc_auc_score(y_test, rf_base.predict_proba(X_test)[:, 1])
auc_lr_opt  = roc_auc_score(y_test, lr_opt.predict_proba(X_test)[:, 1])
auc_rf_opt  = roc_auc_score(y_test, rf_opt.predict_proba(X_test)[:, 1])

print('=== Resumo Final Sprint 3 ===')
print(f'Ganho ROC-AUC — Regressão Logística : {auc_lr_opt - auc_lr_base:+.4f}  (base={auc_lr_base:.4f} → opt={auc_lr_opt:.4f})')
print(f'Ganho ROC-AUC — Random Forest        : {auc_rf_opt - auc_rf_base:+.4f}  (base={auc_rf_base:.4f} → opt={auc_rf_opt:.4f})')

melhor = 'Random Forest' if auc_rf_opt >= auc_lr_opt else 'Regressão Logística'
auc_melhor = max(auc_lr_opt, auc_rf_opt)
print(f'\nMelhor modelo otimizado: {melhor}  (ROC-AUC = {auc_melhor:.4f})')
print(f'\nParâmetros LR: {grid_lr.best_params_}')
print(f'Parâmetros RF: {grid_rf.best_params_}')

## 12. Salvar Modelos Otimizados

In [ ]:
import os
model_dir = os.path.join(BASE_DIR, 'models')
os.makedirs(model_dir, exist_ok=True)

joblib.dump(lr_opt, os.path.join(model_dir, 'modelo_lr_otimizado.pkl'))
joblib.dump(rf_opt, os.path.join(model_dir, 'modelo_rf_otimizado.pkl'))

print('Modelos otimizados salvos:')
print('  • models/modelo_lr_otimizado.pkl')
print('  • models/modelo_rf_otimizado.pkl')
print('\nSprint 3 concluída!')